# PyDS Library Demo

This notebook demonstrates the complete machine learning workflow using the **pyds** library - a Python client for the DeepChem Server API.

## What is `pyds`?

``pyds`` is a Python client for the DeepChem Server API and it provides a clean, object-oriented interface for:

- **Settings Management**: Centralized configuration for profiles, projects, and server connections
- **Data Operations**: Upload and manage molecular datasets
- **ML Primitives**: Featurization, training, evaluation, and inference
- **Workflow Integration**: Seamless chaining of operations for complete ML pipelines

## Prerequisites

1. **DeepChem Server** must be running (default: http://localhost:8000)
2. **Install pyds**: `pip install -e .` (from the pyds directory)
3. **Sample Data**: This demo uses molecular data with SMILES strings

For detailed installation and configuration, see the [documentation](https://deep-forest-sciences-deepchem-server.readthedocs-hosted.com/en/latest/).


## 1. Import Libraries and Check Installation


In [1]:
import pyds
from pyds import Settings, Data
from pyds.primitives import Featurize, Train, Evaluate, Infer, TVTSplit
import pandas as pd
import json

print(pyds.__version__)

0.1.0-alpha.1


## 2. Configuration

Configure your profile, project, and server connection. Settings are automatically saved to `.pyds.settings.json`.


In [2]:
settings = Settings()

# Configure your profile and project
settings.set_profile("demo_user")
settings.set_project("demo_project")
settings.set_base_url("http://localhost:8000")

print(settings)


Settings(profile='demo_user', project='demo_project', base_url='http://localhost:8000')


In [3]:
from pyds.base import BaseClient

try:
    client = BaseClient(settings)
    health_status = client.healthcheck()
    print(f"Server connection successful!")
    print(f"Server status: {health_status}")
except Exception as e:
    print(f"Server connection failed: {e}")


Server connection successful!
Server status: {'status': 'ok'}


## 3. Data Upload

Upload your molecular dataset to the DeepChem datastore. The server returns a `dataset_address` that identifies your uploaded data.


In [4]:
# Create sample molecular data for demonstration
# In practice you would upload your own CSV file

sample_data = {
    'smiles': [
        'CCO',
        'CC(C)O',
        'CC(C)(C)O',
        'c1ccccc1',
        'Cc1ccccc1',
        'CCc1ccccc1',
        'O=C(O)c1ccccc1',
        'CC(=O)O',
        'CCCCO',
        'CCCCCO',
        'CCN',
        'CCC',
        'CCCC',
        'CCCCC',
        'CCCCCC',
        'c1ccc(O)cc1',
        'c1ccc(N)cc1',
        'c1ccc(C)cc1C',
        'CC(C)C',
        'CCC(C)C',
        'CC(C)CC',
        'CCCCN',
        'CCCCCN',
        'CCO[CH2]',
        'CC(=O)N',
        'CCC(=O)O',
        'CCCC(=O)O',
        'c1ccc2ccccc2c1',
        'c1ccc(Cl)cc1',
        'c1ccc(F)cc1'
    ],
    'target': [0.5, 0.6, 0.7, 0.3, 0.4, 0.5, 0.8, 0.9, 0.6, 0.7, 0.4, 0.2, 0.3, 0.4, 0.5, 0.7, 0.6, 0.5, 0.3, 0.4, 0.3, 0.5, 0.6, 0.4, 0.8, 0.7, 0.8, 0.2, 0.3, 0.4]
}

# Create DataFrame and save to CSV
df = pd.DataFrame(sample_data)
csv_file = "sample_molecular_data.csv"
df.to_csv(csv_file, index=False)

print("Sample dataset created:")
print(df.head())
print(f"\nDataset saved as: {csv_file}")


Sample dataset created:
      smiles  target
0        CCO     0.5
1     CC(C)O     0.6
2  CC(C)(C)O     0.7
3   c1ccccc1     0.3
4  Cc1ccccc1     0.4

Dataset saved as: sample_molecular_data.csv


In [5]:
data_client = Data(settings)

upload_response = data_client.upload_data(
    file_path=csv_file,
    filename="demo_molecular_data.csv",
    description="Demo dataset for pyds tutorial"
)

dataset_address = upload_response['dataset_address']

print(f"Dataset uploaded successfully!")
print(f"Upload response: {json.dumps(upload_response, indent=2)}")


Dataset uploaded successfully!
Upload response: {
  "dataset_address": "deepchem://demo_user/demo_project/demo_molecular_data.csv"
}


## 4. Featurization

Transform raw molecular data (SMILES strings) into machine learning features using DeepChem's featurizers.


In [6]:
featurize_client = Featurize(settings)

featurize_response = featurize_client.run(
    dataset_address=dataset_address,
    featurizer="ecfp",
    output="demo_featurized_data",
    dataset_column="smiles",
    feat_kwargs={
        "radius": 2,
        "size": 1024
    },
    label_column="target"
)

featurized_address = featurize_response['featurized_file_address']

print(f"Featurization completed!")
print(f"Response: {json.dumps(featurize_response, indent=2)}")


Featurization completed!
Response: {
  "featurized_file_address": "deepchem://demo_user/demo_project/demo_featurized_data"
}


## 5. Data Splitting (Optional)

Split your dataset into train, validation, and test sets using various splitter types.


In [7]:
split_client = TVTSplit(settings)

split_response = split_client.run(
    splitter_type="random",
    dataset_address=featurized_address,
    frac_train=0.6,
    frac_valid=0.2,
    frac_test=0.2
)

split_data = split_response["train_valid_test_split_results_address"]

print(f"Dataset split completed!")
print(f"Split response: {json.dumps(split_response, indent=2)}")

train_dataset = split_data[0]
valid_dataset = split_data[1]
test_dataset = split_data[2]

Dataset split completed!
Split response: {
  "train_valid_test_split_results_address": [
    "deepchem://demo_user/demo_project/demo_featurized_data_train",
    "deepchem://demo_user/demo_project/demo_featurized_data_valid",
    "deepchem://demo_user/demo_project/demo_featurized_data_test"
  ]
}


## 6. Model Training

Train a machine learning model on your featurized data. pyds supports various model types including scikit-learn models and DeepChem neural networks.


In [8]:
train_client = Train(settings)

train_response = train_client.run(
    dataset_address=train_dataset,
    model_type="random_forest_regressor",
    model_name="demo_rf_model",
    init_kwargs={
        "n_estimators": 100,
        "random_state": 42,
        "max_depth": 10
    },
    train_kwargs={}
)

model_address = train_response['trained_model_address']

print(f"Model training completed!")
print(f"Training response: {json.dumps(train_response, indent=2)}")


Model training completed!
Training response: {
  "trained_model_address": "deepchem://demo_user/demo_project/demo_rf_model"
}


## 7. Model Evaluation

Evaluate your trained model's performance using various metrics on test data.


In [9]:
evaluate_client = Evaluate(settings)

evaluate_response = evaluate_client.run(
    dataset_addresses=[test_dataset],
    model_address=model_address,
    metrics=[
        "pearson_r2_score",
        "rms_score",
        "mae_error"
    ],
    output_key="demo_evaluation",
    is_metric_plots=False
)

eval_address = evaluate_response['evaluation_result_address']

print(f"Model evaluation completed!")
print(f"Evaluation results address: {eval_address}")
print(f"Metrics evaluated: R², RMS, MAE")
print(f"Evaluation response: {json.dumps(evaluate_response, indent=2)}")


Model evaluation completed!
Evaluation results address: deepchem://demo_user/demo_project/demo_evaluation.json
Metrics evaluated: R², RMS, MAE
Evaluation response: {
  "evaluation_result_address": "deepchem://demo_user/demo_project/demo_evaluation.json"
}


Contents in `demo_evaluation.json`:

```json
{
    "deepchem://demo_user/demo_project/demo_featurized_data_test": 
        {
            "pearson_r2_score": 0.9773284614252786,
            "rms_score": 0.05534738777334759,
            "mae_score": 0.041333333333333146
        }
}
```

## 8. Inference

Run predictions on new data using your trained model.


In [10]:
infer_client = Infer(settings)

infer_response = infer_client.run(
    model_address=model_address,
    data_address=test_dataset,
    output="demo_predictions",
    dataset_column="smiles",
)

inference_address = infer_response['inference_results_address']

print(f"Inference completed!")
print(f"Predictions address: {inference_address}")
print(f"Inference response: {json.dumps(infer_response, indent=2)}")


Inference completed!
Predictions address: deepchem://demo_user/demo_project/demo_predictions.csv
Inference response: {
  "inference_results_address": "deepchem://demo_user/demo_project/demo_predictions.csv"
}


Contents in `demo_predictions.csv`:

```csv
X,y_preds
Cc1ccccc1,0.40999999999999986
CC(C)CC,0.39299999999999957
CC(=O)N,0.779
```

## 9. Summary

### Complete Workflow Overview

You've successfully completed a full machine learning pipeline using pyds:

1. **Configuration** - Set up profile, project, and server connection
2. **Data Upload** - Upload molecular dataset to datastore
3. **Featurization** - Transform SMILES to ML features (ECFP)
4. **Data Splitting** - Split into train/validation/test sets
5. **Model Training** - Train Random Forest Regressor
6. **Model Evaluation** - Evaluate with regression metrics
7. **Inference** - Generate predictions on new data

### Key Concepts

- **Dataset Addresses**: All operations use addresses to reference data in the datastore
- **Primitives**: Modular components for each ML task
- **Settings**: Centralized configuration with persistent storage
- **Error Handling**: Comprehensive validation and error reporting


---


> ### Accessing Datastore Files
>
> **Note:** The DeepChem server currently does not support downloading files directly through the API. This feature will be added very soon.
>
> **Workaround:** To access generated files (models, featurized datasets, predictions, etc.) from the datastore, you can mount the container's data directory as a Docker volume:
>
> 1. Update your `docker-compose.yml` to include a volume mount:
>
>    ```yaml
>    services:
>      deepchem-server:
>        volumes:
>          - ./<your_local_directory>:/opt/deepchem_server_app/data
>    ```
>
> 2. Or use the `-v` flag when running Docker directly:
>
>    ```bash
>    docker run -v ./<your_local_directory>:/opt/deepchem_server_app/data -p 8000:8000 deepchem-server
>    ```
>
> The datastore default location inside the container is as follows:
>
> - **Container path**: `/opt/deepchem_server_app/data/{profile_name}/{project_name}/`
> - **Example**: `/opt/deepchem_server_app/data/demo_user/demo_project/`
>
> All generated artifacts (featurized datasets with `.cdc` cards, trained models with `.cmc` cards, predictions, etc.) will be accessible in your local mounted directory.
